# FormulaLM — QLoRA Fine-Tuning (Qwen3-1.7B, Non-Thinking, Unsloth)

Ready-to-run training notebook for a **free Google Colab or Kaggle T4 GPU (16 GB VRAM)**.
Supports both **v1 (frozen baseline: `config/train_qlora.yaml`)** and **v2 (`config/train_qlora_v2.yaml`, default below)**.

## v2 Changes vs Frozen v1 (`v1-frozen`)
- **Data (`train_v2.jsonl`, 9,000 verified examples on the 20 training sheets only):** adds train-only phrasings for per-unit ratios, implicit column headers, unquoted multi-word category literals, numeric `YEAR()` vs `TEXT()`, plus 2 new general ratio templates, rebalanced to ~30% D1 / 35% D2 / 35% D3. `val.jsonl` and `test_synthetic.jsonl` stay bit-for-bit frozen.
- **Hyperparameters (`config/train_qlora_v2.yaml`, chosen from v1's validation loss curve):** `num_train_epochs: 2 → 1` (563 steps, ~16 min on T4), `learning_rate: 2e-4 → 1e-4`, `weight_decay: 0.01 → 0.05`, `warmup_ratio: 0.05 → 0.08`, `eval_steps/save_steps: 100 → 50` (`load_best_model_at_end: true` strictly on `val.jsonl`), keeping 4-bit QLoRA (`r=16`, `alpha=32`), non-thinking mode, and `train_on_responses_only`.
- See [`docs/v2_changelog.md`](../docs/v2_changelog.md) for the full log of data vs hyperparameter changes.

In [ ]:
# 1. Check GPU (must be NVIDIA T4 / CUDA GPU, NOT TPU or CPU)
import torch

assert torch.cuda.is_available(), 'No CUDA GPU detected! Switch runtime to T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0), '| VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 2. Environment Setup (Run **2A for Google Colab** OR **2B for Kaggle**)

In [ ]:
# 2A. GOOGLE COLAB ONLY — Install verified Unsloth Qwen3 stack & optionally mount Google Drive
MOUNT_DRIVE = True
if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print('Drive mount skipped (normal in VS Code/IDE Colab extension):', exc)

%pip install -q sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer pyyaml formulas openpyxl pandas
%pip install -q --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
!unsloth install-kernels
%pip install -q --no-deps --upgrade "torchao>=0.16.0"
%pip install -q transformers==4.57.6
%pip install -q --no-deps trl==0.22.2

In [ ]:
# 2B. KAGGLE ONLY — Run this cell instead of 2A when on Kaggle
RUN_KAGGLE_SETUP = False
if RUN_KAGGLE_SETUP:
    import os
    try:
        from kaggle_secrets import UserSecretsClient
        _ks = UserSecretsClient()
        for _key in ('GITHUB_TOKEN', 'HF_TOKEN'):
            try:
                os.environ[_key] = _ks.get_secret(_key)
                print(f'Loaded Kaggle secret: {_key}')
            except Exception:
                pass
    except Exception as exc:
        print('kaggle_secrets unavailable:', exc)
    %pip install -q torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
    %pip install -q unsloth pyyaml formulas openpyxl pandas
    %pip install -q --no-deps --upgrade "torchao>=0.16.0"
    %pip install -q transformers==4.57.6
    %pip install -q --no-deps trl==0.22.2

In [ ]:
# 3. Clone / pull FormulaLM repo and build verified v2 training split + frozen val/test splits
import getpass, os, pathlib, subprocess, sys

DATASET_VERSION = 'v2'  # 'v2' builds train_v2.jsonl (9k) + frozen val.jsonl & test_synthetic.jsonl
IS_KAGGLE = pathlib.Path('/kaggle/working').is_dir()
REPO_DIR = pathlib.Path('/kaggle/working/FormulaLM' if IS_KAGGLE else '/content/FormulaLM')

if not REPO_DIR.exists():
    token = os.environ.get('GITHUB_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            token = None
    if not token:
        token = getpass.getpass('GitHub PAT for abubakarp789/FormulaLM (hidden, empty if public): ').strip()
    url = f'https://{token}@github.com/abubakarp789/FormulaLM.git' if token else 'https://github.com/abubakarp789/FormulaLM.git'
    subprocess.run(['git', 'clone', url, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

%cd {REPO_DIR}
if str(REPO_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'src'))

!python -m src.data.build_dataset --version {DATASET_VERSION} --workers 2

In [ ]:
# 4. Optional quick 200-example smoke test for v2 (~1-2 min on T4)
CONFIG_PATH = 'config/train_qlora_v2.yaml'
RUN_NAME = 'qlora_v2'

!python scripts/train_qlora.py --config {CONFIG_PATH} --smoke --run-name {RUN_NAME}

print('=== Smoke test loss_log.csv ===')
!cat results/train_{RUN_NAME}_smoke/loss_log.csv

In [ ]:
# 5. Full QLoRA v2 Training Run (9,000 train_v2 / 500 val, 1 epoch = 563 steps, ~15-18 min on T4)
# Auto-resumes from Drive/Kaggle checkpoints if session disconnects. Frozen v1 stays untouched.
import pathlib

CONFIG_PATH = 'config/train_qlora_v2.yaml'
RUN_NAME = 'qlora_v2'

if pathlib.Path('/kaggle/working').is_dir():
    OUTPUT_DIR = f'/kaggle/working/formulalm_outputs/{RUN_NAME}'
elif pathlib.Path('/content/drive/MyDrive').is_dir():
    OUTPUT_DIR = f'/content/drive/MyDrive/FormulaLM/{RUN_NAME}'
else:
    OUTPUT_DIR = f'outputs/{RUN_NAME}'

print('Config:', CONFIG_PATH, '| Output:', OUTPUT_DIR)
!python scripts/train_qlora.py \
  --config {CONFIG_PATH} \
  --run-name {RUN_NAME} \
  --output-dir {OUTPUT_DIR} \
  --resume auto

In [ ]:
# 6. Inspect training/validation loss log & run summary
import pathlib, pandas as pd

res_dir = pathlib.Path(f'results/train_{RUN_NAME}')
print('=== run_summary.json ===')
print((res_dir / 'run_summary.json').read_text())

print('=== loss_log.csv ===')
df_loss = pd.read_csv(res_dir / 'loss_log.csv')
print(df_loss.to_string(index=False))

In [ ]:
# 7. Optional: Push v2 LoRA adapter to HF Hub + push lightweight logs to GitHub
PUSH_ADAPTER_TO_HF = False
HF_REPO_ID = 'abubakar56/formulalm-qwen3-1.7b-lora-v2'

if PUSH_ADAPTER_TO_HF:
    import getpass, os
    from huggingface_hub import HfApi
    hf_token = os.environ.get('HF_TOKEN')
    if not hf_token:
        try:
            from google.colab import userdata
            hf_token = userdata.get('HF_TOKEN')
        except Exception:
            hf_token = getpass.getpass('Hugging Face write token (hidden): ').strip()
    api = HfApi(token=hf_token)
    api.create_repo(HF_REPO_ID, exist_ok=True, repo_type='model')
    api.upload_folder(folder_path=f'{OUTPUT_DIR}/adapter', repo_id=HF_REPO_ID, repo_type='model')
    print('Uploaded v2 adapter to https://huggingface.co/' + HF_REPO_ID)

PUSH_LOGS_TO_GITHUB = True
if PUSH_LOGS_TO_GITHUB:
    !git config user.name "abubakarp789"
    !git config user.email "abubakarp789@gmail.com"
    !git add results/train_{RUN_NAME}*
    !git commit -m "results: add QLoRA v2 training and validation loss logs"
    !git pull --rebase origin main
    !git push origin main